<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border-bottom:1px solid rgba(148,163,184,0.35);padding-bottom:12px;margin-bottom:14px;">
<h1 style="text-align:right;color:inherit;margin:5px 0;">هفتهٔ ۱۵ | ارزیابی مدل‌های طبقه‌بندی (Classification Evaluation)</h1>
</div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<ul><li>ساخت و خواندن Confusion Matrix و چهار حالت TP/TN/FP/FN</li>
<li>Accuracy، Precision، Recall، Specificity، F1، Balanced Accuracy و مسئلهٔ عدم‌توازن کلاس‌ها</li>
<li>Threshold Tuning بر اساس هزینهٔ خطاها، بدون استفاده از Test برای انتخاب</li>
<li>منحنی Precision–Recall، منحنی ROC، ROC-AUC و Average Precision</li>
<li>پیاده‌سازی معیارها از صفر با NumPy و گزارش نهایی یک مسئلهٔ صنعتی</li></ul>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<ol>
<li><strong></strong> Confusion Matrix و Accuracy </li>
<li><strong></strong> Precision/Recall/F1/Specificity و Class Imbalance</li>
<li><strong></strong> Threshold، هزینهٔ خطا، Precision–Recall Curve و انتخاب آستانه</li>
<li><strong></strong> ROC، TPR/FPR، AUC و تفسیر منحنی‌ها </li>
<li><strong></strong> پروژهٔ موتورهای صنعتی و تمرین From Scratch </li></ol>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:1.8;color:inherit;">
<h2 style="text-align:right;color:inherit;margin-bottom:5px;">آماده‌سازی | کتابخانه‌ها و بازسازی دادهٔ موتور</h2><p>داده‌ها مصنوعی‌اند و صرفاً برای آموزش روش ارزیابی تولید شده‌اند؛ به دیتای واقعی صنعت نسبت داده نمی‌شوند.</p></div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, classification_report, precision_recall_curve,
    roc_curve, roc_auc_score, average_precision_score
)

SEED = 42
np.set_printoptions(precision=3, suppress=True)
print("Dependencies loaded.")

In [ ]:
# The same synthetic motor generation mechanism used in Week 14.
rng_motor = np.random.default_rng(2026)
n_motor = 420
temperature_c = np.clip(rng_motor.normal(78, 11, n_motor), 48, 112)
vibration_mm_s = np.clip(
    rng_motor.normal(4.3, 1.5, n_motor) + 0.012 * (temperature_c - 78),
    0.7, 9.5
)
latent_score = -1.0 + 0.075 * (temperature_c - 75) + 0.78 * (vibration_mm_s - 4)
simulated_risk = 1 / (1 + np.exp(-latent_score))
needs_inspection = rng_motor.binomial(1, simulated_risk)

motor_df = pd.DataFrame({
    "temperature_c": temperature_c,
    "vibration_mm_s": vibration_mm_s,
    "needs_inspection": needs_inspection
})
display(motor_df.head())
print("Overall label counts:", motor_df["needs_inspection"].value_counts().sort_index().to_dict())

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #ef4444;background:rgba(239,68,68,0.06);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">چرا سه مجموعه؟</h3>
<p><strong>Train</strong> برای یادگیری وزن‌ها، <strong>Validation</strong> برای تصمیم‌هایی مانند انتخاب Threshold و <strong>Test</strong> برای ارزیابی نهاییِ یک‌باره استفاده می‌شود. اگر با Test آستانه را تنظیم کنیم، نتیجهٔ گزارش نهایی خوش‌بینانه خواهد بود.</p>
</div></div>

In [ ]:
X_motor = motor_df[["temperature_c", "vibration_mm_s"]].to_numpy()
y_motor = motor_df["needs_inspection"].to_numpy()

X_dev, X_test, y_dev, y_test = train_test_split(
    X_motor, y_motor, test_size=0.20, random_state=SEED, stratify=y_motor
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=SEED, stratify=y_dev
)

motor_model = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000, random_state=SEED))
])
motor_model.fit(X_train, y_train)  # Already covered in Week 14; no retraining lesson.
positive_col = int(np.flatnonzero(motor_model.named_steps["logistic"].classes_ == 1)[0])
val_prob = motor_model.predict_proba(X_val)[:, positive_col]
test_prob = motor_model.predict_proba(X_test)[:, positive_col]
val_pred_05 = (val_prob >= 0.5).astype(int)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Positive rates:", {k: round(float(np.mean(v)), 3) for k,v in
                          {"train":y_train,"validation":y_val,"test":y_test}.items()})
assert len(X_train) == 252 and len(X_val) == 84 and len(X_test) == 84

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:1.8;color:inherit;">
<h2 style="text-align:right;color:inherit;margin-bottom:5px;"> Confusion Matrix و Accuracy</h2><p>از دید مسئول تعمیرات، «هشدار غلط» و «خرابی از دست‌رفته» دو نوع اشتباه متفاوت‌اند.</p></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">سؤال شهودی: ۹۵٪ دقت، اما هیچ خرابی‌ای پیدا نشده!</h3>
<p>فرض کن ۲۰۰ موتور داریم؛ ۱۹۰ موتور سالم و ۱۰ موتور نیازمند بازرسی. اگر مدلی به <strong>تمام ۲۰۰ موتور بگوید سالم</strong>، Accuracy آن ۹۵٪ است! اما هر ۱۰ مورد نیازمند بازرسی را از دست داده است. بنابراین قبل از Accuracy باید انواع پیش‌بینی درست و غلط را بشناسیم.</p>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">قرارداد برچسب‌ها در تمام نوت‌بوک</h3>
<p>در همهٔ محاسبات، کلاس <strong>۱ = نیاز به بازرسی</strong> و کلاس <strong>۰ = بدون هشدار بازرسی</strong> است. در Confusion Matrix سطرها <strong>واقعیت</strong> و ستون‌ها <strong>پیش‌بینی</strong> هستند.</p>
</div></div>

$$
C=\begin{bmatrix}
TN & FP\\
FN & TP
\end{bmatrix}
\qquad\text{Rows: actual }[0,1],\quad\text{Columns: predicted }[0,1]
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">خواندن چهار خانه</h3>
<ul>
<li><strong>TP:</strong> موتور واقعاً نیازمند بازرسی بوده و هشدار دریافت کرده است.</li>
<li><strong>TN:</strong> موتور واقعاً بدون نیاز به بازرسی بوده و هشدار دریافت نکرده است.</li>
<li><strong>FP:</strong> هشدار غیرضروری؛ مدل به‌اشتباه هشدار داده است.</li>
<li><strong>FN:</strong> بازرسیِ از دست‌رفته؛ موتور نیازمند بازرسی بوده ولی مدل هشدار نداده است.</li>
</ul><p>در کاربرد صنعتی معمولاً FN می‌تواند پرهزینه‌تر باشد، اما میزان واقعی هزینه‌ها را باید از مسئله شناخت.</p>
</div></div>

In [ ]:
# A completely transparent hand-checkable example (not the motor dataset).
small_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
small_pred = np.array([1, 1, 0, 1, 0, 1, 0, 0, 0, 0])
small_cm = confusion_matrix(small_true, small_pred, labels=[0, 1])
print("Rows=actual; columns=predicted:\n", small_cm)
ConfusionMatrixDisplay(small_cm, display_labels=["No flag (0)", "Inspection (1)"]).plot(
    cmap="Blues", values_format="d", colorbar=False
)
plt.title("10 motors: confusion matrix")
plt.tight_layout(); plt.show()

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">Accuracy یعنی چه؟</h3>
<p>Accuracy نسبت کل پیش‌بینی‌های درست به کل نمونه‌هاست؛ درست یعنی <code>TP + TN</code>. اما وقتی کلاس مثبت نادر باشد، Accuracy ممکن است واقعیت خطر را پنهان کند.</p>
</div></div>

$$
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}
$$

In [ ]:
cm_val = confusion_matrix(y_val, val_pred_05, labels=[0,1])
tn, fp, fn, tp = cm_val.ravel()
print(f"TN={tn} | FP={fp} | FN={fn} | TP={tp}")
print("Validation Accuracy:", round(accuracy_score(y_val, val_pred_05), 3))
fig, ax = plt.subplots(figsize=(5.8,4.6))
ConfusionMatrixDisplay(cm_val, display_labels=["No flag", "Inspection"]).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
ax.set_title("Motor inspections | validation | threshold=0.50")
plt.tight_layout(); plt.show()

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #10b981;background:rgba(16,185,129,0.08);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">✍️ Confusion Matrix از صفر با NumPy</h3>
<p>تابعی بنویس که از دو آرایهٔ <code>y_true</code> و <code>y_pred</code> چهار عدد را با ترتیب <code>(TN, FP, FN, TP)</code> برگرداند. استفاده از <code>sklearn.metrics</code> داخل تابع ممنوع است.</p>
</div></div>

In [ ]:
def confusion_counts_numpy(y_true, y_pred):
    """TODO: Return (TN, FP, FN, TP) using only NumPy."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    # TODO: Count the four combinations with boolean masks.
    return None

scratch_result = confusion_counts_numpy(small_true, small_pred)
print("Your result:", scratch_result)
print("Expected reference:", tuple(small_cm.ravel()))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">معیار بررسی خروجی</h3>
<p>برای مثال کوچک باید <code>TN=5، FP=1، FN=1، TP=3</code> باشد. مجموع خانه‌ها باید برابر ۱۰ شود.</p>
</div></div>

In [ ]:
# Accuracy trap: all-negative prediction on an imbalanced dataset.
rare_true = np.r_[np.zeros(190, dtype=int), np.ones(10, dtype=int)]
rare_pred = np.zeros_like(rare_true)
print("All-negative Accuracy:", accuracy_score(rare_true, rare_pred))
print("Number of missed positives:", int(np.sum((rare_true == 1) & (rare_pred == 0))))
print("Confusion matrix:\n", confusion_matrix(rare_true, rare_pred, labels=[0,1]))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:1.8;color:inherit;">
<h2 style="text-align:right;color:inherit;margin-bottom:5px;"> Precision، Recall، F1، Specificity و Class Imbalance</h2><p>دیگر فقط نمی‌پرسیم «چند درصد درست بود؟»؛ می‌پرسیم «چه نوع خطایی را تحمل می‌کنیم؟»</p></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">تفاوت دو سؤال کلیدی</h3>
<p><strong>Precision:</strong> از بین موتورهایی که هشدار گرفته‌اند، واقعاً چند درصد نیاز به بازرسی داشته‌اند؟</p>
<p><strong>Recall (Sensitivity / TPR):</strong> از بین تمام موتورهای واقعاً نیازمند بازرسی، چند درصد را پیدا کرده‌ایم؟</p>
<p>Precision از «مجموع هشدارها» شروع می‌کند، Recall از «مجموع موارد مثبت واقعی».</p>
</div></div>

$$
\mathrm{Precision}=\frac{TP}{TP+FP}\qquad
\mathrm{Recall}=\frac{TP}{TP+FN}
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">Specificity و FPR</h3>
<p><strong>Specificity (TNR)</strong> نسبت موتورهای واقعاً بدون نیاز به بازرسی است که درست منفی تشخیص داده شده‌اند. <strong>FPR</strong> سهم هشدارهای غیرضروری در بین همهٔ نمونه‌های واقعاً منفی است؛ مکمل Specificity.</p>
</div></div>

$$
\mathrm{Specificity}=\frac{TN}{TN+FP}
\qquad\mathrm{FPR}=\frac{FP}{FP+TN}=1-\mathrm{Specificity}
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">چرا F1 میانگین هارمونیک است؟</h3>
<p>اگر یکی از Precision یا Recall خیلی کوچک باشد، عملکرد کلی هشداردهی خوب نیست. میانگین هارمونیک برخلاف میانگین حسابی اجازه نمی‌دهد یک مقدار خیلی بالا به‌آسانی مقدار خیلی پایین را بپوشاند.</p>
</div></div>

$$
F_1=\frac{2PR}{P+R}=\frac{2TP}{2TP+FP+FN}
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">Balanced Accuracy برای داده نامتوازن</h3>
<p>Balanced Accuracy میانگین Recall کلاس مثبت و Recall کلاس منفی است؛ به همین دلیل در مسئله‌های نامتوازن می‌تواند گویاتر از Accuracy خام باشد.</p>
</div></div>

$$
\mathrm{Balanced\ Accuracy}=\frac{\mathrm{Recall}+\mathrm{Specificity}}{2}
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #ef4444;background:rgba(239,68,68,0.06);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">حالت مخرج صفر</h3>
<p>اگر مدل هیچ هشدار مثبتی صادر نکند، کسر Precision مخرج صفر دارد. در Scikit-learn می‌توان قرارداد <code>zero_division=0</code> را انتخاب کرد؛ یعنی در این وضعیت مقدار محاسباتی Precision را صفر گزارش کنیم. این یک <strong>قرارداد محاسباتی</strong> است، نه تعریف ریاضی کسر ۰/۰.</p>
</div></div>

In [ ]:
def safe_divide(num, den):
    return float(num / den) if den != 0 else 0.0

p_hand = safe_divide(tp, tp+fp)
r_hand = safe_divide(tp, tp+fn)
s_hand = safe_divide(tn, tn+fp)
f1_hand = safe_divide(2*p_hand*r_hand, p_hand+r_hand)
print(pd.Series({
    "Accuracy": safe_divide(tp+tn, tn+fp+fn+tp),
    "Precision": p_hand, "Recall": r_hand,
    "Specificity": s_hand, "F1": f1_hand,
    "Balanced Accuracy": (r_hand+s_hand)/2
}).round(3))
print("\nSklearn classification report (Validation):")
print(classification_report(y_val, val_pred_05,
                            target_names=["No flag", "Inspection"], zero_division=0))

In [ ]:
print("All-negative baseline:")
print("Accuracy:", round(accuracy_score(rare_true, rare_pred), 3))
print("Precision:", precision_score(rare_true, rare_pred, zero_division=0))
print("Recall:", recall_score(rare_true, rare_pred, zero_division=0))
print("F1:", f1_score(rare_true, rare_pred, zero_division=0))
print("Balanced Accuracy:", balanced_accuracy_score(rare_true, rare_pred))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #10b981;background:rgba(16,185,129,0.08);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">✍️ معیارهای طبقه‌بندی از صفر</h3>
<p>تابع <code>classification_metrics_numpy</code> را فقط با آرایه‌های صفر و یک و NumPy تکمیل کن. خروجی باید شامل <code>accuracy</code>، <code>precision</code>، <code>recall</code>، <code>specificity</code>، <code>f1</code>، <code>balanced_accuracy</code> باشد. مخرج صفر را با قرارداد صفر مدیریت کن.</p>
</div></div>

In [ ]:
def classification_metrics_numpy(y_true, y_pred):
    """TODO: Compute binary classification metrics with NumPy only."""
    # TODO 1: Count TN, FP, FN, TP (you may reuse your previous function).
    # TODO 2: Compute accuracy, precision, recall, specificity, F1.
    # TODO 3: Compute balanced accuracy; handle zero denominator safely.
    return None

metric_result = classification_metrics_numpy(y_val, val_pred_05)
print("Your metrics:", metric_result)
print("Reference F1:", round(f1_score(y_val, val_pred_05), 4))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">معیار بررسی خروجی</h3>
<p>وقتی خروجی را تکمیل کردی، مقدار F1 باید با <code>f1_score(y_val, val_pred_05)</code> و Balanced Accuracy با تابع کتابخانه‌ای یکسان شود.</p>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">آزمایش فکری: کدام مدل بهتر است؟</h3>
<p>مدل A: Precision=0.90، Recall=0.30. مدل B: Precision=0.60، Recall=0.85. اگر تشخیص‌ندادن یک دستگاه خطرناک باشد، B ممکن است بهتر باشد؛ اگر توقف ناخواسته بسیار پرهزینه باشد، شرایط می‌تواند عوض شود. بدون <strong>هزینهٔ خطا و هدف عملیاتی</strong> جواب قطعی نداریم.</p>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:1.8;color:inherit;">
<h2 style="text-align:right;color:inherit;margin-bottom:5px;"> Threshold، هزینهٔ خطا و Precision–Recall Curve</h2><p>تنظیم آستانه بدون تغییر وزن‌های مدل: تصمیم تجاری/عملیاتی روی نمرهٔ احتمالی.</p></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">Threshold با آموزش دوباره فرق دارد</h3>
<p>مدل برای هر موتور یک احتمال برمی‌گرداند. Threshold به ما می‌گوید از چه احتمالی بالاتر هشدار صادر کنیم. در نتیجه با عوض‌کردن Threshold فقط <strong>تصمیم نهایی</strong> تغییر می‌کند؛ ضرایب Logistic Regression ثابت می‌مانند.</p>
</div></div>

$$
\hat y_t(\mathbf x)=\begin{cases}
1 & \hat p(\mathbf x)\ge t\\
0 & \hat p(\mathbf x)<t
\end{cases}
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">یک اصل دربارهٔ تغییر آستانه</h3>
<p>با پایین‌آوردن آستانه، تعداد هشدارها هرگز کم نمی‌شود؛ TP و FP هرگز کم نمی‌شوند؛ FN و TN هرگز زیاد نمی‌شوند. ولی <strong>Precision الزاماً یکنواخت نیست</strong>؛ چون کیفیت هشدارهای تازه اضافه‌شده می‌تواند متفاوت باشد.</p>
</div></div>

In [ ]:
def evaluate_threshold(y_true, positive_prob, threshold):
    pred = (np.asarray(positive_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "threshold": float(threshold),
        "alerts": int(pred.sum()),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }

threshold_demo = pd.DataFrame([
    evaluate_threshold(y_val, val_prob, t) for t in [0.30,0.50,0.70]
]).round(3)
display(threshold_demo)

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">تبدیل خطا به هزینهٔ عملیاتی</h3>
<p>برای شبیه‌سازی آموزشی فرض می‌کنیم هزینهٔ هر <strong>FN هشت واحد</strong> و هزینهٔ هر <strong>FP یک واحد</strong> است؛ TN و TP در این تابع هزینه صفر در نظر گرفته شده‌اند. این اعداد واقعیت صنعتی ثابت نیستند و فقط برای گفت‌وگو دربارهٔ انتخاب آستانه استفاده می‌شوند.</p>
</div></div>

$$
\mathrm{Cost}(t)=8\,FN(t)+1\,FP(t)
$$

In [ ]:
# Select the threshold ONLY on validation, never on test.
threshold_grid = np.round(np.arange(0.05, 0.951, 0.025), 3)
threshold_rows = []
for t in threshold_grid:
    row = evaluate_threshold(y_val, val_prob, t)
    row["cost"] = 8 * row["FN"] + row["FP"]
    threshold_rows.append(row)
threshold_df = pd.DataFrame(threshold_rows)

selected_row = threshold_df.sort_values(["cost", "threshold"], ascending=[True,True]).iloc[0]
selected_threshold = float(selected_row["threshold"])
print("Selected threshold (validation cost):", selected_threshold)
display(threshold_df.sort_values("cost").head(8).round(3))

fig, axes = plt.subplots(1,2,figsize=(13,4.3))
axes[0].plot(threshold_df.threshold,threshold_df.cost,marker=".",label="8 FN + 1 FP")
axes[0].axvline(selected_threshold,color="darkorange",ls="--",label="Selected on validation")
axes[0].set(xlabel="Threshold",ylabel="Illustrative cost",title="Validation operational cost")
axes[0].legend(); axes[0].grid(alpha=.25)
axes[1].plot(threshold_df.threshold,threshold_df.precision,label="Precision")
axes[1].plot(threshold_df.threshold,threshold_df.recall,label="Recall")
axes[1].plot(threshold_df.threshold,threshold_df.f1,label="F1")
axes[1].axvline(selected_threshold,color="darkorange",ls="--")
axes[1].set(xlabel="Threshold",ylabel="Metric",ylim=(-.03,1.04),title="Threshold trade-offs (validation)")
axes[1].legend(); axes[1].grid(alpha=.25)
plt.tight_layout();plt.show()

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">منحنی Precision–Recall چه چیزی را نشان می‌دهد؟</h3>
<p>اگر آستانه را در همهٔ موقعیت‌های ممکن جابه‌جا کنیم، برای هر تصمیم یک نقطه با مختصات (Recall, Precision) خواهیم داشت. در نمودار، محور افقی Recall و محور عمودی Precision است.</p><p>در مسائل با کلاس مثبت نادر، منحنی PR معمولاً از Accuracy گویاتر است؛ خط مبنای طبیعیِ Precision برای رتبه‌بندی تصادفی تقریباً <strong>نرخ شیوع کلاس مثبت</strong> است.</p>
</div></div>

$$
\mathrm{PR\ Curve}:\quad x=\mathrm{Recall}(t),\quad y=\mathrm{Precision}(t)
$$

In [ ]:
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_val, val_prob)
val_ap = average_precision_score(y_val, val_prob)
val_prevalence = float(np.mean(y_val))
fig, ax = plt.subplots(figsize=(7,5))
ax.step(pr_recall, pr_precision, where="post",label=f"Model, AP={val_ap:.3f}")
ax.axhline(val_prevalence,ls="--",color="gray",label=f"Prevalence={val_prevalence:.3f}")
for thr in [0.30,0.50,0.70]:
    d=evaluate_threshold(y_val,val_prob,thr)
    ax.scatter(d["recall"], d["precision"],s=55)
    ax.annotate(f"t={thr:.1f}",(d["recall"],d["precision"]),xytext=(5,6),textcoords="offset points")
ax.set(xlabel="Recall", ylabel="Precision", xlim=(-.03,1.03), ylim=(-.03,1.03),
       title="Precision–Recall curve | validation")
ax.legend();ax.grid(alpha=.25);plt.tight_layout();plt.show()
print("Average Precision (AP) =",round(val_ap,4))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #ef4444;background:rgba(239,68,68,0.06);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">AP با مساحت ذوزنقه‌ای چه فرقی دارد؟</h3>
<p><strong>Average Precision</strong> در Scikit-learn با جمع‌زدن تغییرات Recall و Precision در نقاط منحنی محاسبه می‌شود؛ لزوماً برابر مساحت حاصل از درون‌یابی خطی با <code>np.trapz</code> نیست. برای گزارش این درس از <code>average_precision_score</code> استفاده می‌کنیم.</p>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #10b981;background:rgba(16,185,129,0.08);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">✍️ آزمایش عملی Threshold Sweep از صفر</h3>
<p>برای Thresholdهای <code>[0.3, 0.5, 0.7]</code> فقط با NumPy تعداد هشدارها، FN، FP و هزینهٔ <code>8*FN+FP</code> را محاسبه کن. خروجی را به شکل DataFrame بساز. از Test استفاده نکن.</p>
</div></div>

In [ ]:
def threshold_sweep_numpy(y_true, probabilities, thresholds, fn_cost=8, fp_cost=1):
    """TODO: A threshold sweep using only NumPy and Pandas."""
    rows = []
    for t in thresholds:
        # TODO: Convert probability to predicted label.
        # TODO: Count alerts, FP, FN, and calculate cost.
        pass
    # TODO: Return a DataFrame with the specified columns.
    return pd.DataFrame(rows)

my_sweep=threshold_sweep_numpy(y_val,val_prob,[0.30,0.50,0.70])
print("Your rows:",len(my_sweep),"| Expected rows: 3")
display(my_sweep)

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">معیار بررسی خروجی</h3>
<p>هزینه‌های محاسبه‌شده باید با جدول مرجع و معادله <code>8×FN + FP</code> سازگار باشند.</p>
</div></div>

In [ ]:
%pip install ipywidgets

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional interactive slider not available. Run: %pip install ipywidgets")
else:
    from IPython.display import display as _display
    def live_threshold_view(threshold=0.5):
        pred=(val_prob>=threshold).astype(int)
        cm=confusion_matrix(y_val,pred,labels=[0,1])
        fig,ax=plt.subplots(figsize=(4.6,3.4))
        ConfusionMatrixDisplay(cm,display_labels=["No flag","Inspection"]).plot(
            ax=ax,cmap="Blues",values_format="d",colorbar=False)
        ax.set_title(f"Validation | threshold={threshold:.2f}")
        plt.tight_layout();plt.show()
        print("Precision:",round(precision_score(y_val,pred,zero_division=0),3),
              "Recall:",round(recall_score(y_val,pred,zero_division=0),3),
              "FP/FN:",int(cm[0,1]),int(cm[1,0]))
    _slider=widgets.FloatSlider(value=0.5,min=0.05,max=0.95,step=0.05,
                                description="Threshold",continuous_update=False)
    _display(widgets.interactive(live_threshold_view,threshold=_slider))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:1.8;color:inherit;">
<h2 style="text-align:right;color:inherit;margin-bottom:5px;"> ROC Curve، TPR/FPR و AUC</h2><p>مقایسهٔ قدرت رتبه‌بندی مدل در آستانه‌های مختلف، مستقل از یک Threshold ثابت.</p></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">سؤال شهودی ROC</h3>
<p>یک مدل خوب معمولاً به موتورهای واقعاً نیازمند بازرسی نمره/احتمال بالاتری می‌دهد. منحنی ROC نشان می‌دهد وقتی آستانه را جابه‌جا می‌کنیم، چند مثبت واقعی را پیدا می‌کنیم و در مقابل چند منفی واقعی را اشتباهاً هشدار می‌دهیم.</p>
</div></div>

$$
\mathrm{TPR}=\frac{TP}{TP+FN}=\mathrm{Recall}
\qquad\mathrm{FPR}=\frac{FP}{FP+TN}
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">مختصات ROC</h3>
<p>هر نقطهٔ ROC برابر <code>(FPR, TPR)</code> است. نقطهٔ (۰،۱) نقطهٔ ایدئال است: همهٔ مثبت‌ها کشف شده‌اند و هیچ هشدار اشتباهی نداریم. خط قطری معمولاً عملکرد رتبه‌بندی تصادفی را نشان می‌دهد.</p>
</div></div>

$$
\mathrm{ROC}:\qquad (x,y)=(\mathrm{FPR}(t),\mathrm{TPR}(t))
$$

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_val,val_prob)
roc_auc = roc_auc_score(y_val,val_prob)
fig,axes=plt.subplots(1,2,figsize=(12.8,4.9))
axes[0].plot(fpr,tpr,linewidth=2,label=f"Motor model | AUC={roc_auc:.3f}")
axes[0].plot([0,1],[0,1],ls="--",color="gray",label="Random ranking")
for t in [0.30,0.50,0.70]:
    cm=confusion_matrix(y_val,(val_prob>=t).astype(int),labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    xx=fp/(fp+tn) if (fp+tn) else 0.0
    yy=tp/(tp+fn) if (tp+fn) else 0.0
    axes[0].scatter(xx,yy,s=55)
    axes[0].annotate(f"t={t:.1f}",(xx,yy),xytext=(4,7),textcoords="offset points")
axes[0].set(xlabel="False Positive Rate",ylabel="True Positive Rate",
            xlim=(-.03,1.03),ylim=(-.03,1.03),title="ROC | validation")
axes[0].legend();axes[0].grid(alpha=.25)
axes[1].hist(val_prob[y_val==0],bins=np.linspace(0,1,16),alpha=.65,label="Actual 0")
axes[1].hist(val_prob[y_val==1],bins=np.linspace(0,1,16),alpha=.65,label="Actual 1")
axes[1].set(xlabel="Estimated positive probability",ylabel="Count",title="Probability score distributions")
axes[1].legend();axes[1].grid(alpha=.25)
plt.tight_layout();plt.show()
print("Validation ROC-AUC:",round(roc_auc,3))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">AUC دقیقاً چه معنا دارد؟</h3>
<p>ROC-AUC اندازهٔ قدرت <strong>رتبه‌بندی</strong> مدل است: اگر تصادفی یک نمونهٔ مثبت و یک نمونهٔ منفی برداریم، AUC تقریباً احتمال آن است که مدل به مثبت نمره بالاتری بدهد (در تساوی‌ها نصف امتیاز). AUC=0.5 عملکرد رتبه‌بندی تصادفی و AUC=1 جداسازی کامل است. AUC بالا تضمین نمی‌کند احتمال‌ها <strong>خوب کالیبره</strong> شده باشند یا یک Threshold عملیاتی خوب انتخاب شده باشد.</p>
</div></div>

$$
\mathrm{AUC}=P(s(X^+)>s(X^-)) + \tfrac12 P(s(X^+)=s(X^-))
$$

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">ROC یا PR؟</h3>
<p>برای مسئله‌های با کلاس مثبت نادر، ROC-AUC ممکن است خوب باشد ولی Precision عملیاتی پایین بماند. بنابراین هر دو منحنی را کنار هم بخوان و به نرخ وقوع کلاس مثبت دقت کن. ROC بیشتر به TPR/FPR و PR مستقیماً به کیفیت هشدارها توجه می‌کند.</p>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">چرا شیوع کلاس مهم است؟</h3>
<p>با TPR و FPR ثابت، Precision هنوز به نرخ وقوع واقعی مثبت‌ها وابسته است. مثلاً اگر Recall=0.8 و FPR=0.1 باشد، در مسئله‌ای با فقط ۵٪ مثبت واقعی تعداد هشدارهای اشتباه می‌تواند بخش بزرگی از تمام هشدارها را تشکیل دهد.</p>
</div></div>

$$
\mathrm{Precision}=\frac{\mathrm{TPR}\cdot\pi}{\mathrm{TPR}\cdot\pi+\mathrm{FPR}\cdot(1-\pi)}\quad\quad \pi=P(Y=1)
$$

In [ ]:
# Same TPR / FPR; sharply different precision as prevalence changes.
example_tpr, example_fpr = 0.8, 0.1
prevalences = [0.50, 0.20, 0.05, 0.01]
prevalence_table = pd.DataFrame({
    "positive_prevalence": prevalences,
    "precision": [example_tpr*pi/(example_tpr*pi+example_fpr*(1-pi)) for pi in prevalences]
})
display(prevalence_table.round(3))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #10b981;background:rgba(16,185,129,0.08);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">✍️ ROC-AUC از صفر با زوج‌های مثبت/منفی | چالش تکمیلی</h3>
<p>برای یک آرایهٔ کوچک از برچسب‌های صفر و یک و نمره‌های پیش‌بینی، AUC را از راه مقایسهٔ تمام زوج‌های (مثبت، منفی) حساب کن. امتیاز نمرهٔ مساوی برابر ۰٫۵ است. در این تمرین از <code>roc_auc_score</code> داخل تابع استفاده نکن.</p>
</div></div>

In [ ]:
def auc_pairwise_numpy(y_true, scores):
    """TODO: AUC from every positive-negative score pair, NumPy only."""
    y_true=np.asarray(y_true)
    scores=np.asarray(scores,dtype=float)
    positives=scores[y_true==1]
    negatives=scores[y_true==0]
    # TODO: Compare each positive score with each negative score.
    # TODO: Score 1 for positive > negative, 0.5 for a tie, 0 otherwise.
    # TODO: Average the pairwise scores.
    return None

pair_y=np.array([0,0,1,1])
pair_s=np.array([.1,.4,.35,.8])
print("Your AUC:",auc_pairwise_numpy(pair_y,pair_s))
print("Reference:",roc_auc_score(pair_y,pair_s))

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #0ea5e9;background:rgba(14,165,233,0.07);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">معیار بررسی خروجی</h3>
<p>در مثال کوچک، AUC باید <code>0.75</code> شود؛ برای مجموعهٔ Validation نیز باید با <code>roc_auc_score</code> برابر باشد.</p>
</div></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:1.8;color:inherit;">
<h2 style="text-align:right;color:inherit;margin-bottom:5px;"> گزارش نهایی ارزیابی Classification</h2><p>اکنون آستانه از Validation انتخاب شده؛ وقت آن است که تنها یک‌بار ارزیابی نهایی را روی Test ببینیم.</p></div>

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #ef4444;background:rgba(239,68,68,0.06);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<p>مدل Logistic Regression از قبل Fit شده و در این هفته آن را از صفر دوباره آموزش نمی‌دهیم. داده‌های Test را برای انتخاب Threshold، طراحی ویژگی‌ها یا مقایسهٔ چند تصمیمِ منتخب استفاده نکرده‌ایم. اکنون آستانهٔ انتخاب‌شده را ثابت نگه می‌داریم.</p>
</div></div>

In [ ]:
# Final evaluation: threshold was frozen using Validation.
test_pred_final = (test_prob >= selected_threshold).astype(int)
final_cm = confusion_matrix(y_test, test_pred_final, labels=[0,1])
final_tn,final_fp,final_fn,final_tp = final_cm.ravel()
final_report = pd.Series({
    "Threshold (validation-selected)": selected_threshold,
    "Test samples": len(y_test),
    "Alerts": int(test_pred_final.sum()),
    "TN": int(final_tn), "FP": int(final_fp),
    "FN": int(final_fn), "TP": int(final_tp),
    "Accuracy": accuracy_score(y_test,test_pred_final),
    "Precision": precision_score(y_test,test_pred_final,zero_division=0),
    "Recall": recall_score(y_test,test_pred_final,zero_division=0),
    "Specificity": final_tn/(final_tn+final_fp) if (final_tn+final_fp) else 0.,
    "F1": f1_score(y_test,test_pred_final,zero_division=0),
    "Balanced Accuracy": balanced_accuracy_score(y_test,test_pred_final),
    "ROC-AUC": roc_auc_score(y_test,test_prob),
    "Average Precision": average_precision_score(y_test,test_prob),
    "Illustrative cost (8 FN + FP)": 8*final_fn + final_fp,
})
display(final_report.round(4).to_frame("Final test result"))
fig,ax=plt.subplots(figsize=(5.5,4.6))
ConfusionMatrixDisplay(final_cm,display_labels=["No flag", "Inspection"]).plot(
    ax=ax,cmap="Blues",values_format="d",colorbar=False
)
ax.set_title(f"Final Test | frozen threshold={selected_threshold:.3f}")
plt.tight_layout();plt.show()

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #8b5cf6;background:rgba(139,92,246,0.075);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<h3 style="margin:0 0 8px 0;text-align:right;color:inherit;">دیدن نمونه‌های خطا</h3>
<p>هر عدد در Confusion Matrix یک نمونهٔ واقعی در مجموعه آزمون دارد. به‌جای قضاوت کلی، دما، لرزش و احتمالِ موتورهای FP و FN را ببین و دلیل هزینهٔ عملیاتی هرکدام را توضیح بده.</p>
</div></div>

In [ ]:
inspection_df=pd.DataFrame({
    "temperature_c":X_test[:,0],
    "vibration_mm_s":X_test[:,1],
    "actual":y_test,
    "estimated_probability":test_prob,
    "prediction":test_pred_final
})
inspection_df["error_type"]=np.select(
    [ (inspection_df.actual==1)&(inspection_df.prediction==0),
      (inspection_df.actual==0)&(inspection_df.prediction==1)],
    ["FN: missed inspection","FP: unnecessary alert"],
    default="correct"
)
print("Test error counts:")
display(inspection_df.error_type.value_counts().to_frame("n"))
display(inspection_df[inspection_df.error_type!="correct"].head(12).round(3))

fig, ax=plt.subplots(figsize=(8,5.2))
for label,col,mark in [("correct","steelblue","o"),
                       ("FP: unnecessary alert","darkorange","x"),
                       ("FN: missed inspection","crimson","X")]:
    sub=inspection_df[inspection_df.error_type==label]
    if len(sub):
        ax.scatter(sub.temperature_c,sub.vibration_mm_s,
                   label=f"{label} (n={len(sub)})",marker=mark,s=65,alpha=.85)
ax.set(xlabel="Temperature (°C)",ylabel="Vibration (mm/s)",
       title="Where does the selected decision rule fail? (Test)")
ax.legend(fontsize=8);ax.grid(alpha=.25);plt.tight_layout();plt.show()

<div dir="rtl" lang="fa" style="direction:rtl;text-align:right;line-height:2;color:inherit;">
<div style="border:1px solid rgba(148,163,184,0.35);border-right:5px solid #10b981;background:rgba(16,185,129,0.08);padding:14px 18px;border-radius:8px;margin:12px 0;color:inherit;">
<p>یک تابع گزارش بنویس که از <code>y_true</code> و <code>positive_prob</code> و <code>threshold</code> خروجی شامل چهار خانهٔ ماتریس، Accuracy/Precision/Recall/F1 و هزینه ۸FN+FP را برگرداند. فقط از NumPy استفاده کن؛ کد نباید Test را برای انتخاب Threshold به کار بگیرد.</p>
</div></div>

In [ ]:
def final_classification_report_numpy(y_true, positive_prob, threshold, fn_cost=8, fp_cost=1):
    """TODO: Create the complete classification report using NumPy only."""
    # TODO: Convert probabilities to labels using the given frozen threshold.
    # TODO: Count TN, FP, FN, TP.
    # TODO: Compute accuracy, precision, recall, specificity, f1.
    # TODO: Add operational cost = fn_cost * FN + fp_cost * FP.
    return None

print("Your final report:", final_classification_report_numpy(y_test,test_prob,selected_threshold))
print("Sklearn reference F1:", round(f1_score(y_test,test_pred_final,zero_division=0),4))